# Experiment 2: N-scaling sweep

Follow-up to `bv_dd_replication.ipynb` (Experiment 1). Experiment 1 found
all 5 variants (`free`, `dd`=UR-14, `xx`, `xpxm`, `xy4`) statistically
indistinguishable on real hardware at N=6 (avg success probability
0.839-0.853). Its idle-time-utilization diagnostic explained why: idle
windows at N=6 are mostly too short for these DD sequences to do anything
-- UR-14 covered only 16.5% of idle time and was silently skipped outright
on 5 of 7 active qubits.

`build_bv_circuit`'s marked qubits all CNOT sequentially onto one shared
ancilla, so increasing N directly lengthens the idle windows the other
data qubits sit in while waiting their turn. This notebook sweeps N to
test whether that actually improves DD utilization and, in turn, whether
DD's benefit over `free` becomes visible once it is.

See `Experiment Plan.md` for the full experiment log and design-decisions
record shared across all notebooks in this project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bv_dd_lib as bv

## 1. Connect to IBM Quantum and pick a backend

Same connection logic as Experiment 1, now factored into
`bv.connect_and_pick_backend` so it isn't copy-pasted per notebook. Reads
credentials from the `MY_TOKEN`/`MY_INSTANCE` environment variables (see
`.env`), falling back to a previously saved `QiskitRuntimeService` account
if they aren't set.

In [ ]:
PREFERRED_BACKENDS = ["ibm_fez", "ibm_marrakesh"]
service, backend = bv.connect_and_pick_backend(PREFERRED_BACKENDS)

## 2. DD sequences under test

Same five variants as Experiment 1: our own UR-14 construction (Appendix
F) plus the three sequences IBM Runtime's `SamplerV2` ships natively
(`XX`, `XpXm`, `XY4`). See Experiment 1 for the derivation and identity
verification of each -- this notebook reuses `bv.ur_n_sequence` /
`bv.ibm_builtin_dd_sequence` directly rather than re-deriving them.

In [ ]:
UR_LENGTH = 14
ur14 = bv.ur_n_sequence(UR_LENGTH)
assert bv.verify_dd_sequence_identity(ur14), "UR-14 sequence does not compose to identity"

builtin_sequences = {name: bv.ibm_builtin_dd_sequence(name) for name in ["XX", "XpXm", "XY4"]}
for name, seq in builtin_sequences.items():
    assert bv.verify_dd_sequence_identity(seq), f"{name} sequence does not compose to identity"

dd_sequences = {"dd": ur14, **builtin_sequences}
variant_cols = ["free"] + list(dd_sequences)
print(f"Variants under test: {variant_cols}")

## 3. `skip_reset_qubits` check (at N=6, against Experiment 1's own data)

Experiment 1's idle-utilization diagnostic found qubits where the
predicted DD coverage didn't match what was actually left in the circuit
-- consistent with `skip_reset_qubits=True` (`PadDynamicalDecoupling`'s
default) declining to pad the idle window immediately following each
qubit's initialization, regardless of how long it is. Before committing
the sweep to a `skip_reset_qubits` setting, check directly at N=6 whether
turning it off recovers those windows for UR-14 (the longest, and
therefore most idle-window-sensitive, sequence).

In [ ]:
N_CHECK = 6
labels_check = bv.all_representative_oracles(N_CHECK)
qc_check = bv.build_bv_circuit(labels_check[-1])  # heaviest oracle, same convention as Experiment 1's `example`

isa_check = bv.to_isa_circuit(qc_check, backend)
free_check = bv.schedule_only(isa_check, backend)

dd_default = bv.add_dynamical_decoupling(isa_check, backend, ur14, skip_reset_qubits=True)
dd_no_skip = bv.add_dynamical_decoupling(isa_check, backend, ur14, skip_reset_qubits=False)

report_default = bv.dd_utilization_report(free_check, dd_default, backend, ur14)
report_no_skip = bv.dd_utilization_report(free_check, dd_no_skip, backend, ur14)

comparison = report_default[["actual_pulse_covered_dt"]].rename(
    columns={"actual_pulse_covered_dt": "skip_reset_qubits=True"}
).join(
    report_no_skip[["actual_pulse_covered_dt"]].rename(columns={"actual_pulse_covered_dt": "skip_reset_qubits=False"})
)
display(comparison)

improves = comparison["skip_reset_qubits=False"].sum() > comparison["skip_reset_qubits=True"].sum()
SKIP_RESET_QUBITS = not improves
print(f"-> using skip_reset_qubits={SKIP_RESET_QUBITS} for the sweep below")

## 4. N sweep: build, transpile, and locally validate every variant at each N

`N_VALUES` includes 6 as a continuity check against Experiment 1's own
numbers. For each N: build the n+1 representative oracles, transpile
every variant via `bv.build_variant_circuits`, run the ideal correctness
gate (must be 1.0 across the board -- a failure here means a
transpilation/DD bug, not a real effect), run the noisy dress rehearsal,
and record the heaviest oracle's idle-utilization summary. Everything in
this section is local simulation -- no QPU time is used.

The noisy simulator is built once (`bv.build_noisy_simulator`) and reused
across every N via `bv.run_on_simulator`, rather than going through
`bv.run_noisy_local` per N -- that would rebuild the (expensive,
backend-sized) noise model once per N value for no reason, since the
calibration snapshot doesn't change across the sweep.

In [ ]:
N_VALUES = [6, 10]
SHOTS = 2000

noisy_sim = bv.build_noisy_simulator(backend)
sweep = {}

for N in N_VALUES:
    labels = bv.all_representative_oracles(N)
    circuits = {label: bv.build_bv_circuit(label) for label in labels}

    per_label = {
        label: bv.build_variant_circuits(circuits[label], backend, dd_sequences, skip_reset_qubits=SKIP_RESET_QUBITS)
        for label in labels
    }

    routed_list = [per_label[label]["routed"] for label in labels]
    variant_lists = {col: [per_label[label][col] for label in labels] for col in variant_cols}
    combined_circuits = [c for col in variant_cols for c in variant_lists[col]]

    ideal_by_variant = bv.split_counts_by_variant(bv.run_ideal(combined_circuits, shots=1000), labels, variant_cols)
    ideal_comparison = bv.compare_variants(ideal_by_variant, labels)
    assert (ideal_comparison[variant_cols] > 0.999).all().all(), \
        f"N={N}: ideal success probability should be 1.0 for all variants -- check DD construction before continuing."

    noisy_counts = bv.run_on_simulator(combined_circuits, noisy_sim, shots=SHOTS)
    noisy_by_variant = bv.split_counts_by_variant(noisy_counts, labels, variant_cols)
    noisy_comparison = bv.compare_variants(noisy_by_variant, labels)

    example = labels[-1]
    non_free_cols = [c for c in variant_cols if c != "free"]
    utilization = bv.dd_utilization_summary(
        per_label[example]["free"],
        {col: per_label[example][col] for col in non_free_cols},
        {col: dd_sequences[col] for col in non_free_cols},
        backend,
    )

    sweep[N] = {
        "labels": labels,
        "routed_list": routed_list,
        "variant_lists": variant_lists,
        "noisy_avg": noisy_comparison[variant_cols].mean(),
        "utilization_pct": utilization["utilization_pct"],
    }
    print(f"N={N}: ideal correctness OK; noisy avg success prob -- " +
          ", ".join(f"{c}: {sweep[N]['noisy_avg'][c]:.3f}" for c in variant_cols))

print("N-sweep local validation complete.")

## 5. Local-sim plots: does DD utilization and benefit grow with N?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for col in variant_cols:
    axes[0].plot(N_VALUES, [sweep[N]["noisy_avg"][col] for N in N_VALUES], marker="o", label=col)
axes[0].set_xlabel("N")
axes[0].set_ylabel("Avg success probability (noisy sim)")
axes[0].set_title("Success probability vs. N")
axes[0].legend()

for col in [c for c in variant_cols if c != "free"]:
    axes[1].plot(N_VALUES, [sweep[N]["utilization_pct"][col] for N in N_VALUES], marker="o", label=col)
axes[1].set_xlabel("N")
axes[1].set_ylabel("Idle-window utilization (%)")
axes[1].set_title("DD utilization vs. N (heaviest oracle)")
axes[1].legend()

fig.suptitle(f"BV-N sweep, N={N_VALUES} -- local noisy simulation ({backend.name} calibration)")
plt.tight_layout()
plt.show()

## 6. QPU budget check

Per-N and cumulative estimates across every N and all 5 variants. This
sweep submits up to `4 jobs x len(N_VALUES)` on hardware (one job for
`free`+`dd` batched together, plus one job per built-in preset, per N --
e.g. 16 jobs for the default 4 N-values) since `options.dynamical_decoupling`
is a per-job Runtime setting, not a per-circuit transpiler pass (see
Experiment 1). Circuit duration also grows with N, so the cumulative
estimate can end up dominated by the largest N even though it's only one
of several. Reduce `N_VALUES` or `SHOTS` above if this prints "OVER
BUDGET".

In [ ]:
QPU_BUDGET_SECONDS = 240  # comfortably under the ~600s/28-day Open Plan quota

per_n_circuits = {
    N: sweep[N]["variant_lists"]["free"] + [c for col in variant_cols if col != "free" for c in sweep[N]["variant_lists"][col]]
    for N in N_VALUES
}

for N in N_VALUES:
    print(f"N={N}: ", end="")
    bv.check_budget(per_n_circuits[N], SHOTS, backend, budget_seconds=QPU_BUDGET_SECONDS)

all_circuits_by_n = [c for N in N_VALUES for c in per_n_circuits[N]]
print("Cumulative across all N: ", end="")
budget_ok = bv.check_budget(all_circuits_by_n, SHOTS, backend, budget_seconds=QPU_BUDGET_SECONDS)

## 7. Real-hardware run

**This cell spends real IBM Quantum QPU quota**, potentially across many
jobs (see the budget check above). Only set `SUBMIT = True` once the
cumulative budget check prints OK and the local-sim plots above show the
mechanism behaving as expected. Same per-N job pattern as Experiment 1:
one `SamplerV2` job for `free`+`dd` (UR-14) batched together, plus one job
per built-in preset (`XX`, `XpXm`, `XY4`), each submitting that N's
*unscheduled* routed ISA circuits (`sweep[N]["routed_list"]`).

`job.result()` blocks until the job completes, which may include real
queue time -- all job IDs are printed so you can reconnect and fetch
results later (`service.job(job_id)`) if the kernel restarts while
queued.

In [ ]:
SUBMIT = False  # flip to True to actually submit to hardware

jobs = {}  # jobs[N] = {"free_dd": Job, "XX": Job, "XpXm": Job, "XY4": Job}
if SUBMIT:
    if not budget_ok:
        raise RuntimeError("Cumulative budget check failed -- reduce N_VALUES or SHOTS before submitting.")

    for N in N_VALUES:
        free_dd_circuits = sweep[N]["variant_lists"]["free"] + sweep[N]["variant_lists"]["dd"]
        job = bv.run_on_hardware(free_dd_circuits, backend, shots=SHOTS)
        print(f"N={N}: submitted job (free + UR-14 DD): {job.job_id()}")

        builtin_jobs = {}
        for name in ["XX", "XpXm", "XY4"]:
            builtin_jobs[name] = bv.run_on_hardware_builtin_dd(sweep[N]["routed_list"], backend, shots=SHOTS, sequence_type=name)
            print(f"N={N}: submitted job ({name} built-in DD): {builtin_jobs[name].job_id()}")

        jobs[N] = {"free_dd": job, **builtin_jobs}
else:
    print("SUBMIT is False -- skipping real-hardware submission.")

## 8. Results: success probability vs. N (real hardware)

In [ ]:
if jobs:
    hw_avg = {}
    for N in N_VALUES:
        labels = sweep[N]["labels"]
        free_dd_result = jobs[N]["free_dd"].result()
        free_dd_counts = bv.counts_from_sampler_result(free_dd_result, 2 * len(labels))
        hw_free = dict(zip(labels, free_dd_counts[: len(labels)]))
        hw_dd = dict(zip(labels, free_dd_counts[len(labels):]))

        hw_builtin = {}
        for name in ["XX", "XpXm", "XY4"]:
            bw_result = jobs[N][name].result()
            bw_counts = bv.counts_from_sampler_result(bw_result, len(labels))
            hw_builtin[name] = dict(zip(labels, bw_counts))

        hw_comparison = bv.compare_variants({"free": hw_free, "dd": hw_dd, **hw_builtin}, labels)
        hw_avg[N] = hw_comparison[variant_cols].mean()
        print(f"N={N}: " + ", ".join(f"{c}: {hw_avg[N][c]:.3f}" for c in variant_cols))

    fig, ax = plt.subplots(figsize=(7, 4))
    for col in variant_cols:
        ax.plot(N_VALUES, [hw_avg[N][col] for N in N_VALUES], marker="o", label=col)
    ax.set_xlabel("N")
    ax.set_ylabel("Avg success probability")
    ax.set_title(f"BV-N sweep on {backend.name} (real hardware)")
    ax.legend()
    plt.show()
else:
    print("No hardware jobs were submitted in this run -- set SUBMIT = True in Section 7 to run on real hardware.")

## Next steps

Update `Experiment Plan.md`'s Experiment 2 entry with the actual job IDs,
the `skip_reset_qubits` decision from Section 3, and the observed findings
once this run completes. If a clear DD benefit emerges at higher N,
revisit Experiment 3 (twirling/readout mitigation) and Experiment 4
(custom `initial_layout`) next -- see `Experiment Plan.md` for their
current scope.